### answer these six questions in Spark SQL against Gold

#### (a) Operations wants to know which machine types tie up the longest. Average rental duration by asset type, returned rentals only.  >> Required output: asset_type, avg_duration_days (2 dp), rentals

In [ ]:
df_rental = spark.table("gold.gold_fact_rental")

df_rental.createOrReplaceTempView("view_rental")

In [ ]:
df = spark.sql("""
SELECT * FROM view_rental;
""")

In [18]:
%%sql   

SELECT 
    asset_id,
    ROUND(avg(rental_duration_days),2) as avg_duration_days,
    count(*) as rental
FROM view_rental
WHERE is_returned = 'true'
GROUP BY asset_id
ORDER BY avg_duration_days DESC;


StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 20, Finished, Available, Finished, False)

<Spark SQL result set with 156 rows and 3 fields>

#### (b) Rhea wants to see which depots are running hot. Fleet utilisation by depot, using asset_days_in_window over fleet_size times thirty. >>> Required output: depot_name, fleet_size, asset_days_used (1 dp), utilisation_pct (1dp).


In [9]:
df_depot = spark.table('gold.gold_dim_depot')
df_depot.createOrReplaceTempView('view_depot')

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 11, Finished, Available, Finished, False)

In [10]:
df = spark.sql("""
SELECT 
    d.depot_name,
    d.fleet_size,
    round(sum(r.asset_days_in_window),1) as asset_days_used,
    ROUND(
        100.0 * SUM(r.asset_days_in_window)/ NULLIF(d.fleet_size * 30, 0),
        1
    ) AS utilisation_pct
FROM view_depot d 
LEFT JOIN view_rental r 
ON d.depot_code = r.depot_code
GROUP BY d.depot_name,d.fleet_size
""")

display(df)

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 12, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e7505c4-a110-41a4-b4f7-8a3ded4f885b)

####  (c) Finance wants the revenue split by how customers pay. Revenue by payer type. >>> Required output: payer_type, bills, revenue (0 dp), avg_bill (0 dp)

In [11]:
df_billing = spark.table('gold.gold_fact_billing')
df_billing.createOrReplaceTempView('view_billing')

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 13, Finished, Available, Finished, False)

In [12]:
df = spark.sql("""

SELECT 
    payer_type,
    count(*)as bills,
    round(sum(amount_inr),0)as revenue,
    round(avg(amount_inr),0)as avg_bill
FROM view_billing
GROUP BY payer_type
ORDER BY revenue DESC;
""")

display(df)

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 14, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, be4c66a8-b5f1-4207-9c7e-0f49481ce07a)

#### (d) Arun wants revenue by depot, the number no single system holds. Join billing to rentals to the depot dimension. >>> Required output: depot_name, bills, revenue (0 dp), revenue_per_rental (0 dp).

In [13]:
df = spark.sql("""

    WITH depot_revenue AS(
        SELECT 
        d.depot_name,
        count(distinct(b.bill_id))as bills,
        sum(b.amount_inr)as revenue,
        count(distinct(r.rental_id))as rental
    FROM view_billing b
    JOIN view_rental r 
        ON b.rental_id = r.rental_id
    JOIN view_depot d 
        ON r.depot_code = d.depot_code 
    GROUP BY d.depot_name
    )

    SELECT  
        depot_name,
        bills,
        round(revenue,0)as revenue,
        round(
            revenue / nullif(rental,0),0
        )as revenue_per_rental

    FROM depot_revenue 
    ORDER BY revenue DESC

""")

display(df)

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 15, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 49f209e2-4d20-4675-aa88-b4fa734ec751)

#### (e) The board wants the priority mix. Share of rentals by rental type. >>> Required output: rental_type, rentals, pct (1 dp).

In [15]:
df = spark.sql("""
    SELECT
        r.rental_type,
        ROUND(SUM(b.amount_inr), 0) AS rental,

        ROUND(
            SUM(b.amount_inr) * 100.0
            / SUM(SUM(b.amount_inr)) OVER (),
            1
        ) AS rental_pct

    FROM view_rental r
    JOIN view_billing b
        ON r.rental_id = b.rental_id

    GROUP BY r.rental_type
    ORDER BY rental DESC
""")

display(df)

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e1bcf98b-6f0d-4470-8b30-8936071ff67b)

#### (f) Rhea wants to know what is out right now. Machines currently out by depot. >>> Required output: depot_name, currently_out.


In [16]:
df = spark.sql("""

SELECT 
    d.depot_name,
    count(distinct(r.rental_id))as currently_out
FROM view_depot d 
JOIN view_rental r 
ON d.depot_code =  r.depot_code
WHERE r.is_returned = 'false'
GROUP BY d.depot_name
ORDER BY currently_out DESC;

""")

display(df)

StatementMeta(, b2161783-7871-43df-ae2b-3108a3ce85a4, 18, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 156d6f96-2c09-4512-8c75-280ff40d0068)

#### 